# 0. Setup and samples

We start from the **raw Xenium output** for one kidney slide (ID `0011695`, 480-gene
panel, FFPE). The slide carries **8 tissue samples** from three glomerulonephritis
types (5 ANCA-GN, 1 anti-GBM, 2 lupus/SLE). This notebook reads the slide, splits it
into the 8 samples, and does quality control.

### Setup

In [ ]:
import sys
sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt
import course_utils as cu

sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. Read the raw slide

> **Method note: what Xenium gives you.** Unlike Visium spots, Xenium localises
individual transcripts and assigns them to cells via a segmentation mask. The output
bundle holds a cell x gene matrix, a per-cell table (`cells.parquet`: centroid, area,
counts, and control counts), the transcripts, and morphology images. Here we read the
matrix and cell table. This panel keeps only the 480 genes in the matrix; the
**control** counts (negative-control probes and codewords, used to estimate the noise
floor) live in the cell table, so we build `control_frac` from there.

In [ ]:
adata = cu.read_xenium_raw()      # cell x gene (480-gene panel), cell table in .obs
print(adata)
adata.obs[['total_counts_all', 'control_counts', 'control_frac', 'cell_area']].describe().round(3)

## 2. Split the slide into samples

> **Method note: separating cores by coordinates.** The 8 samples are physically
separate tissue cores on one slide, so they occupy distinct regions of the shared
micron coordinate system. `data/sample_bounding_boxes.csv` gives each sample's box
(derived from an independent, more careful segmentation of the same slide; the cell
barcodes differ but the coordinates match). We assign each raw cell to the box that
contains its centroid. Cells in the gaps between cores (~0.5%) are dropped.

In [ ]:
boxes = cu.load_sample_boxes()
display(boxes)
adata = cu.assign_samples(adata)            # adds .obs['sample','patient_sample_id','disease']
print(adata.shape)
print(adata.obs.groupby('disease', observed=True)['sample'].value_counts())

In [ ]:
# the 8 tissue cores on the slide, coloured by sample
sq.pl.spatial_scatter(adata, color='sample', shape=None, size=1, figsize=(8, 9))

## 3. Quality control

> **Method note: Xenium QC, and what to look for.** Read **transcripts per cell** and
**genes per cell** (too few = an empty or fragmented segment), the **control fraction**
(high = noise / non-specific signal), and **cell area** (very small or very large
segments are suspect). Thresholds are dataset-specific: look at the distributions and
per-sample summaries first, then choose. Always sanity-check that no single sample is
an outlier driven by a technical effect rather than biology.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)
fig, axs = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, col, log in zip(axs, ['total_counts', 'n_genes_by_counts', 'control_frac', 'cell_area'],
                        [True, True, False, False]):
    vals = adata.obs[col].values
    ax.hist(np.log10(vals + 1) if log else vals, bins=60, color='#1f5f8c')
    ax.set_title(('log10 ' if log else '') + col)
fig.tight_layout(); plt.show()

Per-sample QC summary (watch for a sample that stands apart):

In [ ]:
adata.obs.groupby('sample', observed=True).agg(
    n_cells=('total_counts', 'size'),
    median_counts=('total_counts', 'median'),
    median_genes=('n_genes_by_counts', 'median'),
    median_control_frac=('control_frac', 'median'),
    median_area=('cell_area', 'median')).round(3)

> **Exercise.** Choose minimum transcripts-per-cell, minimum genes-per-cell, and a
maximum control fraction, then filter `adata` to the cells that pass. Print how many
cells you keep. (The solution uses 10 counts, 5 genes, 5% control.)

In [ ]:
# Solution: filter low-quality cells and keep the result for later notebooks.
MIN_COUNTS, MIN_GENES, MAX_CONTROL = 10, 5, 0.05
keep = ((adata.obs['total_counts'] >= MIN_COUNTS) &
        (adata.obs['n_genes_by_counts'] >= MIN_GENES) &
        (adata.obs['control_frac'] <= MAX_CONTROL))
print(f'keeping {keep.sum()} of {adata.n_obs} cells')
adata = adata[keep].copy()

## 4. Save for the next notebooks

We write the QC'd object to `results/` (git-ignored). The later notebooks load this
instead of re-reading the raw slide.

In [ ]:
from pathlib import Path
Path('results').mkdir(exist_ok=True)
adata.write('results/slide_0011695_qc.h5ad')
print('saved results/slide_0011695_qc.h5ad', adata.shape)

### Recap

We read the raw slide, split it into the 8 samples by coordinate bounding boxes,
computed Xenium QC metrics, filtered, and saved. Next: a full transcriptomic analysis
of one sample.